In [ ]:
# ============================================================
# MITT THANDAVAPURA COLLEGE Q&A ASSISTANT
# BLACK TEXT / LETTERS + COLORFUL UI
#
# Features:
# - MITT Thandavapura
# - Official website crawler
# - Page-by-page navigation
# - Suggested questions
# - Keyword-based search
# - College background image
# - Colorful UI
# - ALL LETTERS / TEXT BLACK
# - Clearly visible Load Website button
# - College details
# - No Qwen
# - No FAISS
# - No embeddings
# - No Chatbot component
# ============================================================


# ============================================================
# 1. INSTALL PACKAGES
# ============================================================

!pip -q install requests beautifulsoup4 gradio


# ============================================================
# 2. IMPORTS
# ============================================================

import requests
import re
import time
import gradio as gr

from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse, urldefrag
from collections import deque


# ============================================================
# 3. COLLEGE CONFIGURATION
# ============================================================

COLLEGE_NAME = "Maharaja Institute of Technology Thandavapura"

COLLEGE_SHORT_NAME = "MITT"

COLLEGE_URL = "https://mitt.edu.in/"


# ============================================================
# 4. BACKGROUND IMAGE
# ============================================================

BACKGROUND_IMAGE = (
    "https://gopalaswamyinstitutions.in/"
    "wp-content/uploads/2024/03/MITT-Campus.jpg"
)


# ============================================================
# 5. SETTINGS
# ============================================================

MAX_PAGES = 25

REQUEST_TIMEOUT = 12

MIN_TEXT_LENGTH = 80

MAX_CHUNKS_PER_PAGE = 25

CHUNK_WORDS = 120


# ============================================================
# 6. GLOBAL VARIABLES
# ============================================================

website_loaded = False

college_pages = []

website_content = []

visited_urls = set()

base_domain = ""


# ============================================================
# 7. HTTP SESSION
# ============================================================

session = requests.Session()

session.headers.update({

    "User-Agent":
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 "
        "(KHTML, like Gecko) "
        "Chrome/131.0 Safari/537.36",

    "Accept":
        "text/html,application/xhtml+xml,"
        "application/xml;q=0.9,*/*;q=0.8",

    "Accept-Language":
        "en-US,en;q=0.9"

})


# ============================================================
# 8. NORMALIZE URL
# ============================================================

def normalize_url(url):

    if not url:
        return ""

    url = str(url).strip()

    if not url.startswith(
        ("http://", "https://")
    ):
        url = "https://" + url

    try:

        parsed = urlparse(url)

        scheme = parsed.scheme.lower()

        hostname = parsed.hostname

        if not hostname:
            return ""

        hostname = hostname.lower()

        if hostname.startswith("www."):
            hostname = hostname[4:]

        path = parsed.path or "/"

        path = re.sub(
            r"/+",
            "/",
            path
        )

        if path != "/" and path.endswith("/"):
            path = path[:-1]

        return (
            scheme
            + "://"
            + hostname
            + path
        )

    except Exception:

        return ""


# ============================================================
# 9. GET DOMAIN
# ============================================================

def get_domain(url):

    try:

        hostname = urlparse(url).hostname

        if not hostname:
            return ""

        hostname = hostname.lower()

        if hostname.startswith("www."):
            hostname = hostname[4:]

        return hostname

    except Exception:

        return ""


# ============================================================
# 10. SAME DOMAIN
# ============================================================

def same_domain(url):

    domain = get_domain(url)

    if not domain:
        return False

    if not base_domain:
        return False

    return (
        domain == base_domain
        or domain.endswith("." + base_domain)
    )


# ============================================================
# 11. VALID URL
# ============================================================

def valid_url(url):

    if not url:
        return False

    try:

        parsed = urlparse(url)

        if parsed.scheme not in [
            "http",
            "https"
        ]:
            return False

        if not parsed.netloc:
            return False

        path = parsed.path.lower()

        blocked_extensions = (

            ".jpg",
            ".jpeg",
            ".png",
            ".gif",
            ".webp",
            ".svg",
            ".ico",

            ".mp3",
            ".wav",

            ".mp4",
            ".avi",
            ".mov",
            ".mkv",

            ".zip",
            ".rar",
            ".7z",

            ".css",
            ".js",

            ".woff",
            ".woff2",
            ".ttf",

            ".xlsx",
            ".xls",
            ".doc",
            ".docx",
            ".ppt",
            ".pptx",
            ".pdf"
        )

        if path.endswith(
            blocked_extensions
        ):
            return False

        return True

    except Exception:

        return False


# ============================================================
# 12. CLEAN TEXT
# ============================================================

def clean_text(text):

    if not text:
        return ""

    text = text.replace(
        "\xa0",
        " "
    )

    text = re.sub(
        r"\s+",
        " ",
        text
    )

    return text.strip()


# ============================================================
# 13. DOWNLOAD PAGE
# ============================================================

def download_page(url):

    try:

        response = session.get(
            url,
            timeout=REQUEST_TIMEOUT,
            allow_redirects=True
        )

        print(
            f"[{response.status_code}] {url}"
        )

        if response.status_code != 200:
            return None

        content_type = response.headers.get(
            "Content-Type",
            ""
        ).lower()

        if "text/html" not in content_type:
            return None

        if len(response.content) == 0:
            return None

        return response

    except requests.exceptions.Timeout:

        print("TIMEOUT:", url)

        return None

    except requests.exceptions.RequestException as e:

        print(
            "REQUEST ERROR:",
            str(e)
        )

        return None

    except Exception as e:

        print(
            "DOWNLOAD ERROR:",
            str(e)
        )

        return None


# ============================================================
# 14. EXTRACT PAGE
# ============================================================

def extract_page(url, response):

    try:

        soup = BeautifulSoup(
            response.text,
            "html.parser"
        )

        for tag in soup.find_all([
            "script",
            "style",
            "noscript",
            "svg",
            "canvas",
            "iframe",
            "form",
            "nav",
            "footer",
            "header"
        ]):

            tag.decompose()

        title = ""

        if soup.title:

            title = soup.title.get_text(
                " ",
                strip=True
            )

        main = soup.find("main")

        if main:

            text = main.get_text(
                " ",
                strip=True
            )

        else:

            body = soup.find("body")

            if body:

                text = body.get_text(
                    " ",
                    strip=True
                )

            else:

                text = soup.get_text(
                    " ",
                    strip=True
                )

        text = clean_text(text)

        if len(text) < MIN_TEXT_LENGTH:
            return None

        return {

            "url": url,

            "title": title,

            "text": text

        }

    except Exception as e:

        print(
            "EXTRACTION ERROR:",
            str(e)
        )

        return None


# ============================================================
# 15. EXTRACT LINKS
# ============================================================

def extract_links(page_url, response):

    links = []

    try:

        soup = BeautifulSoup(
            response.text,
            "html.parser"
        )

        for tag in soup.find_all("a"):

            href = tag.get("href")

            if not href:
                continue

            href = href.strip()

            if href.startswith(
                (
                    "#",
                    "mailto:",
                    "tel:",
                    "javascript:",
                    "whatsapp:"
                )
            ):
                continue

            absolute = urljoin(
                page_url,
                href
            )

            absolute = urldefrag(
                absolute
            )[0]

            absolute = normalize_url(
                absolute
            )

            if not absolute:
                continue

            if not same_domain(
                absolute
            ):
                continue

            if not valid_url(
                absolute
            ):
                continue

            links.append(
                absolute
            )

    except Exception as e:

        print(
            "LINK ERROR:",
            str(e)
        )

    return list(
        dict.fromkeys(links)
    )


# ============================================================
# 16. CRAWL WEBSITE
# ============================================================

def crawl_website():

    global base_domain
    global college_pages
    global visited_urls

    college_pages = []

    visited_urls = set()

    start_url = normalize_url(
        COLLEGE_URL
    )

    base_domain = get_domain(
        start_url
    )

    queue = deque()

    queue.append(
        start_url
    )

    print()
    print("=" * 70)
    print("STARTING MITT WEBSITE CRAWLER")
    print("=" * 70)

    while (
        queue
        and len(college_pages) < MAX_PAGES
    ):

        current_url = queue.popleft()

        current_url = normalize_url(
            current_url
        )

        if not current_url:
            continue

        if current_url in visited_urls:
            continue

        if not same_domain(
            current_url
        ):
            continue

        if not valid_url(
            current_url
        ):
            continue

        visited_urls.add(
            current_url
        )

        print(
            f"\n[{len(college_pages)+1}/{MAX_PAGES}] "
            f"{current_url}"
        )

        response = download_page(
            current_url
        )

        if response is None:
            continue

        final_url = normalize_url(
            response.url
        )

        if final_url:
            current_url = final_url

        page = extract_page(
            current_url,
            response
        )

        if page:

            college_pages.append(
                page
            )

            print(
                "   ✓ Saved:",
                page["title"]
            )

        links = extract_links(
            current_url,
            response
        )

        for link in links:

            if link in visited_urls:
                continue

            if link in queue:
                continue

            if len(queue) >= MAX_PAGES * 3:
                break

            queue.append(
                link
            )

        time.sleep(
            0.05
        )

    print()
    print("=" * 70)
    print("CRAWLING FINISHED")
    print("=" * 70)

    print(
        "Pages loaded:",
        len(college_pages)
    )

    print(
        "URLs visited:",
        len(visited_urls)
    )

    return college_pages


# ============================================================
# 17. BUILD SEARCH CONTENT
# ============================================================

def build_search_content():

    global website_content

    website_content = []

    for page in college_pages:

        text = page["text"]

        words = text.split()

        start = 0

        chunk_count = 0

        while (
            start < len(words)
            and chunk_count < MAX_CHUNKS_PER_PAGE
        ):

            chunk_words = words[
                start:start + CHUNK_WORDS
            ]

            chunk = " ".join(
                chunk_words
            )

            if len(chunk) >= 80:

                website_content.append({

                    "text": chunk,

                    "url": page["url"],

                    "title": page["title"]

                })

                chunk_count += 1

            start += CHUNK_WORDS

    print(
        "Search chunks:",
        len(website_content)
    )


# ============================================================
# 18. QUESTION TOPICS
# ============================================================

QUESTION_TOPICS = {

    "courses": [
        "course",
        "courses",
        "program",
        "programs",
        "degree",
        "degrees",
        "undergraduate",
        "postgraduate",
        "btech",
        "b.e",
        "be",
        "mtech",
        "mba",
        "mca",
        "phd",
        "engineering"
    ],

    "admission": [
        "admission",
        "admissions",
        "apply",
        "application",
        "enrolment",
        "enrollment"
    ],

    "eligibility": [
        "eligibility",
        "eligible",
        "qualification",
        "qualifications",
        "criteria",
        "requirements"
    ],

    "fees": [
        "fee",
        "fees",
        "tuition",
        "cost",
        "fee structure"
    ],

    "placements": [
        "placement",
        "placements",
        "recruiter",
        "recruiters",
        "company",
        "companies",
        "career",
        "careers",
        "salary",
        "package"
    ],

    "hostel": [
        "hostel",
        "hostels",
        "accommodation",
        "residence",
        "room"
    ],

    "scholarship": [
        "scholarship",
        "scholarships",
        "financial aid",
        "funding"
    ],

    "departments": [
        "department",
        "departments",
        "school",
        "schools",
        "faculty"
    ],

    "campus": [
        "campus",
        "campuses",
        "infrastructure",
        "facility",
        "facilities"
    ],

    "library": [
        "library",
        "libraries",
        "books",
        "reading"
    ],

    "sports": [
        "sports",
        "sport",
        "playground",
        "gym",
        "fitness"
    ],

    "clubs": [
        "club",
        "clubs",
        "student club",
        "student clubs",
        "activities",
        "events"
    ],

    "internship": [
        "internship",
        "internships",
        "training",
        "industrial training"
    ],

    "research": [
        "research",
        "research center",
        "research centre",
        "innovation",
        "phd"
    ],

    "location": [
        "location",
        "located",
        "address",
        "campus address",
        "where is",
        "situated"
    ],

    "about": [
        "about",
        "history",
        "established",
        "vision",
        "mission",
        "known"
    ],

    "facilities": [
        "facility",
        "facilities",
        "laboratory",
        "laboratories",
        "lab",
        "labs",
        "transport",
        "transportation",
        "infrastructure"
    ]

}


# ============================================================
# 19. DETECT TOPIC
# ============================================================

def detect_topic(question):

    q = question.lower()

    scores = {}

    for topic, keywords in QUESTION_TOPICS.items():

        score = 0

        for keyword in keywords:

            if keyword in q:

                score += 1

        scores[topic] = score

    best_topic = max(
        scores,
        key=scores.get
    )

    if scores[best_topic] == 0:

        return "general"

    return best_topic


# ============================================================
# 20. SEARCH CONTENT
# ============================================================

def search_content(question):

    if not website_content:
        return []

    q = question.lower()

    topic = detect_topic(q)

    words = re.findall(
        r"[a-zA-Z]{3,}",
        q
    )

    stop_words = {

        "what",
        "which",
        "where",
        "when",
        "who",
        "how",
        "does",
        "are",
        "is",
        "the",
        "this",
        "that",
        "college",
        "university",
        "please",
        "tell",
        "about",
        "give",
        "me",
        "can",
        "you",
        "for",
        "from",
        "with",
        "have",
        "has",
        "their",
        "there",
        "available",
        "offer",
        "offers",
        "provide",
        "provides"
    }

    keywords = [

        word

        for word in words

        if word not in stop_words

    ]

    topic_keywords = QUESTION_TOPICS.get(
        topic,
        []
    )

    scored = []

    for item in website_content:

        text = item["text"].lower()

        score = 0

        for keyword in keywords:

            if keyword in text:

                score += 2

        for keyword in topic_keywords:

            if keyword in text:

                score += 5

        title = item["title"].lower()

        for keyword in topic_keywords:

            if keyword in title:

                score += 8

        if score > 0:

            scored.append(
                (
                    score,
                    item
                )
            )

    scored.sort(
        key=lambda x: x[0],
        reverse=True
    )

    results = []

    seen = set()

    for score, item in scored:

        key = item["text"][:120]

        if key in seen:
            continue

        seen.add(
            key
        )

        results.append(
            item
        )

        if len(results) >= 8:
            break

    return results


# ============================================================
# 21. SENTENCE EXTRACTION
# ============================================================

def extract_relevant_sentences(
    question,
    results
):

    if not results:
        return []

    topic = detect_topic(
        question
    )

    topic_keywords = QUESTION_TOPICS.get(
        topic,
        []
    )

    question_words = set(
        re.findall(
            r"[a-zA-Z]{4,}",
            question.lower()
        )
    )

    sentences = []

    for item in results:

        text = item["text"]

        parts = re.split(
            r"(?<=[.!?])\s+",
            text
        )

        for sentence in parts:

            sentence = sentence.strip()

            if len(sentence) < 35:
                continue

            lower = sentence.lower()

            score = 0

            for keyword in topic_keywords:

                if keyword in lower:

                    score += 4

            for word in question_words:

                if word in lower:

                    score += 2

            if score > 0:

                sentences.append(
                    (
                        score,
                        sentence,
                        item["url"]
                    )
                )

    sentences.sort(
        key=lambda x: x[0],
        reverse=True
    )

    final_sentences = []

    seen = set()

    for score, sentence, url in sentences:

        normalized = re.sub(
            r"\W+",
            " ",
            sentence.lower()
        ).strip()

        if normalized in seen:
            continue

        seen.add(
            normalized
        )

        final_sentences.append(
            (
                sentence,
                url
            )
        )

        if len(final_sentences) >= 8:
            break

    return final_sentences


# ============================================================
# 22. CLEAN ANSWER
# ============================================================

def clean_answer_sentence(sentence):

    sentence = sentence.strip()

    sentence = re.sub(
        r"\s+",
        " ",
        sentence
    )

    unwanted_phrases = [

        "skip to main content",
        "click here to apply",
        "read more",
        "learn more",
        "cookie policy",
        "privacy policy",
        "terms and conditions",
        "follow us on",
        "all rights reserved",
        "contact us",
        "contact our team",
        "request info",
        "application form"

    ]

    lower_sentence = sentence.lower()

    for phrase in unwanted_phrases:

        if phrase in lower_sentence:

            return ""

    return sentence


# ============================================================
# 23. GENERATE ANSWER
# ============================================================

def generate_answer(
    question,
    results
):

    if not results:

        return (
            "## ❌ Information Not Found\n\n"
            "I could not find relevant information "
            "for this question on the MITT website.\n\n"
            "Please try another suggested question."
        )

    relevant_sentences = extract_relevant_sentences(
        question,
        results
    )

    if not relevant_sentences:

        return (
            "## ❌ Information Not Found\n\n"
            "I could not find a clear answer to this "
            "question on the MITT website."
        )

    answer_lines = []

    used_urls = []

    for sentence, url in relevant_sentences:

        sentence = clean_answer_sentence(
            sentence
        )

        if not sentence:
            continue

        if len(sentence) > 450:

            sentence = sentence[:447] + "..."

        answer_lines.append(
            sentence
        )

        if url not in used_urls:

            used_urls.append(
                url
            )

        if len(answer_lines) >= 5:

            break

    if not answer_lines:

        return (
            "## ❌ Information Not Found\n\n"
            "No useful information was found."
        )

    answer = (
        "## 🎓 Answer\n\n"
    )

    for line in answer_lines:

        answer += (
            "- "
            + line
            + "\n\n"
        )

    if used_urls:

        answer += (
            "---\n\n"
            "### 🌐 Source\n\n"
        )

        answer += (
            used_urls[0]
            + "\n"
        )

    return answer


# ============================================================
# 24. ASK QUESTION
# ============================================================

def ask_question(question):

    question = str(
        question or ""
    ).strip()

    if not question:

        return (
            "## ❓ Select a Question\n\n"
            "Please select a suggested question."
        )

    if not website_loaded:

        return (
            "## ⚠️ Website Not Loaded\n\n"
            "Please click **Load MITT Website** first."
        )

    print()
    print("=" * 70)
    print("QUESTION")
    print("=" * 70)

    print(question)

    topic = detect_topic(
        question
    )

    print(
        "Detected topic:",
        topic
    )

    results = search_content(
        question
    )

    print(
        "Relevant chunks:",
        len(results)
    )

    return generate_answer(
        question,
        results
    )


# ============================================================
# 25. LOAD COLLEGE
# ============================================================

def load_college():

    global website_loaded

    website_loaded = False

    print()
    print("=" * 70)
    print("LOADING MITT WEBSITE")
    print("=" * 70)

    try:

        pages = crawl_website()

        if not pages:

            return (
                "## ❌ Website Could Not Be Loaded\n\n"
                "The MITT website could not be read.\n\n"
                "Please check your internet connection "
                "and try again."
            )

        build_search_content()

        if not website_content:

            return (
                "## ❌ No Content Found\n\n"
                "The website opened, but readable content "
                "could not be extracted."
            )

        website_loaded = True

        return (
            "## 🟢 MITT Website Loaded\n\n"
            "### College\n"
            + COLLEGE_NAME
            + "\n\n"
            "### Pages Loaded\n"
            + str(len(college_pages))
            + "\n\n"
            "### Search Sections\n"
            + str(len(website_content))
            + "\n\n"
            "### Status\n"
            "The Q&A assistant is ready."
        )

    except Exception as e:

        website_loaded = False

        print(
            "LOAD ERROR:",
            str(e)
        )

        return (
            "## ❌ Loading Error\n\n"
            "The website could not be loaded.\n\n"
            "Please try again."
        )


# ============================================================
# 26. SUGGESTED QUESTIONS
# ============================================================

QUESTIONS = [

    "What courses does MITT offer?",
    "What undergraduate programs are available?",
    "What postgraduate programs are available?",
    "What engineering courses are offered?",
    "What is the admission process?",
    "How can I apply for admission?",
    "What are the eligibility requirements?",
    "What is the fee structure?",
    "What are the tuition fees?",
    "What are the placement opportunities?",
    "Which companies recruit students?",
    "What placement support does MITT provide?",
    "What facilities are available on campus?",
    "What hostel facilities are available?",
    "Does MITT have a library?",
    "What sports facilities are available?",
    "What student clubs and activities are available?",
    "What internship opportunities are available?",
    "What scholarships are available?",
    "What departments does MITT have?",
    "What research opportunities are available?",
    "Where is MITT located?",
    "What is MITT known for?",
    "What are the vision and mission of MITT?",
    "What facilities does MITT provide to students?",
    "Is transportation available for students?",
    "What are the campus facilities at MITT?",
    "What are the MBA and MCA programs?",
    "What M.Tech programs are available?",
    "What makes MITT different from other colleges?"

]


# ============================================================
# 27. PAGE NAVIGATION
# ============================================================

def show_page(page_number):

    if not college_pages:

        return (
            "## ⚠️ Website Not Loaded\n\n"
            "Please load the MITT website first."
        )

    try:

        index = int(page_number) - 1

    except Exception:

        index = 0

    if index < 0:
        index = 0

    if index >= len(college_pages):
        index = len(college_pages) - 1

    page = college_pages[index]

    text = page["text"]

    if len(text) > 5000:

        text = text[:5000] + "\n\n..."

    return (
        "## 📄 "
        + page["title"]
        + "\n\n"
        "**Page:** "
        + str(index + 1)
        + " / "
        + str(len(college_pages))
        + "\n\n"
        "**URL:**\n"
        + page["url"]
        + "\n\n"
        "---\n\n"
        + text
    )


def next_page(page_number):

    if not college_pages:

        return page_number

    try:

        current = int(page_number)

    except Exception:

        current = 1

    if current < len(college_pages):

        current += 1

    return current


def previous_page(page_number):

    try:

        current = int(page_number)

    except Exception:

        current = 1

    if current > 1:

        current -= 1

    return current


def select_question(question):

    return question


# ============================================================
# 28. CUSTOM CSS
# ============================================================

CUSTOM_CSS = f"""

/* ============================================================
   GLOBAL
   ============================================================ */

html,
body,
.gradio-container {{

    margin: 0 !important;

    padding: 0 !important;

    background: #050505 !important;

    color: #000000 !important;

    font-family:
        Arial,
        Helvetica,
        sans-serif !important;
}}


/* ============================================================
   BACKGROUND
   ============================================================ */

.gradio-container::before {{

    content: "";

    position: fixed;

    inset: 0;

    background-image:
        linear-gradient(
            rgba(255,255,255,0.35),
            rgba(255,255,255,0.55)
        ),
        url("{BACKGROUND_IMAGE}");

    background-size: cover;

    background-position: center;

    background-repeat: no-repeat;

    z-index: -2;
}}


/* ============================================================
   MAIN CONTAINER
   ============================================================ */

#main-container {{

    max-width: 1250px !important;

    margin: 25px auto !important;

    padding: 28px !important;

    background:
        rgba(255,255,255,0.92) !important;

    border:
        2px solid #111111 !important;

    border-radius:
        24px !important;

    box-shadow:
        0 20px 60px
        rgba(0,0,0,0.65) !important;
}}


/* ============================================================
   HEADER
   ============================================================ */

#college-header {{

    text-align: center !important;

    padding: 35px 20px !important;

    margin-bottom: 20px !important;

    background:
        linear-gradient(
            135deg,
            #00d4ff,
            #7b2cff,
            #ff2d95
        ) !important;

    border:
        2px solid #000000 !important;

    border-radius:
        22px !important;

    box-shadow:
        0 10px 35px
        rgba(0,0,0,0.6) !important;
}}


/* ============================================================
   HEADER TEXT
   ============================================================ */

#college-header h1,
#college-header h2,
#college-header h3,
#college-header p,
#college-header span,
#college-header strong {{

    color:
        #000000 !important;
}}


#college-header h1 {{

    font-size:
        36px !important;

    font-weight:
        900 !important;

    letter-spacing:
        0.5px !important;
}}


#college-header p {{

    font-size:
        16px !important;

    font-weight:
        600 !important;
}}


/* ============================================================
   CARDS
   ============================================================ */

.section-card {{

    background:
        rgba(255,255,255,0.96) !important;

    border:
        2px solid #111111 !important;

    border-radius:
        18px !important;

    padding:
        22px !important;

    margin-top:
        20px !important;

    box-shadow:
        0 8px 30px
        rgba(0,0,0,0.45) !important;
}}


/* ============================================================
   ALL MARKDOWN
   ============================================================ */

.markdown,
.markdown *,
.markdown p,
.markdown li,
.markdown span,
.markdown strong,
.markdown em,
.markdown a,
.markdown h1,
.markdown h2,
.markdown h3,
.markdown h4,
.markdown h5,
.markdown h6 {{

    color:
        #000000 !important;
}}


/* ============================================================
   COLLEGE DETAILS
   ============================================================ */

.details-card {{

    background:
        linear-gradient(
            135deg,
            #ffffff,
            #e8f8ff
        ) !important;

    border-left:
        6px solid #00d4ff !important;
}}


/* ============================================================
   WEBSITE BOX
   ============================================================ */

.website-box {{

    background:
        linear-gradient(
            135deg,
            #00d4ff,
            #a8f5ff
        ) !important;

    color:
        #000000 !important;

    border:
        2px solid #000000 !important;

    border-radius:
        12px !important;

    padding:
        15px !important;

    text-align:
        center !important;

    font-weight:
        800 !important;
}}


/* ============================================================
   LOAD BUTTON
   ============================================================ */

#load-button {{

    display:
        block !important;

    visibility:
        visible !important;

    opacity:
        1 !important;

    width:
        100% !important;

    min-height:
        55px !important;

    background:
        linear-gradient(
            135deg,
            #00c853,
            #00e676
        ) !important;

    color:
        #000000 !important;

    border:
        3px solid #000000 !important;

    border-radius:
        14px !important;

    font-size:
        18px !important;

    font-weight:
        900 !important;

    box-shadow:
        0 8px 25px
        rgba(0,200,83,0.30) !important;
}}


#load-button *,
#load-button span {{

    color:
        #000000 !important;
}}


#load-button:hover {{

    background:
        linear-gradient(
            135deg,
            #69f0ae,
            #00c853
        ) !important;

    transform:
        translateY(-2px) !important;

    box-shadow:
        0 12px 30px
        rgba(0,230,118,0.40) !important;
}}


/* ============================================================
   STATUS
   ============================================================ */

#status-box {{

    margin-top:
        15px !important;

    background:
        #ffffff !important;

    border:
        2px solid #000000 !important;

    border-radius:
        14px !important;

    padding:
        18px !important;
}}


#status-box,
#status-box *,
#status-box h2,
#status-box h3,
#status-box p,
#status-box span {{

    color:
        #000000 !important;
}}


/* ============================================================
   QUESTION BUTTONS
   ============================================================ */

.question-button {{

    background:
        linear-gradient(
            135deg,
            #ffe600,
            #ffb300
        ) !important;

    color:
        #000000 !important;

    border:
        2px solid #000000 !important;

    border-radius:
        12px !important;

    min-height:
        62px !important;

    font-size:
        14px !important;

    font-weight:
        800 !important;

    transition:
        all 0.2s ease !important;
}}


.question-button *,
.question-button span {{

    color:
        #000000 !important;
}}


.question-button:hover {{

    background:
        linear-gradient(
            135deg,
            #ffea00,
            #ff9800
        ) !important;

    color:
        #000000 !important;

    border-color:
        #000000 !important;

    transform:
        translateY(-2px) !important;

    box-shadow:
        0 8px 20px
        rgba(255,152,0,0.30) !important;
}}


/* ============================================================
   INPUT
   ============================================================ */

textarea,
input,
select {{

    background:
        #ffffff !important;

    color:
        #000000 !important;

    border:
        2px solid #000000 !important;

    border-radius:
        12px !important;
}}


textarea *,
input *,
select * {{

    color:
        #000000 !important;
}}


textarea:focus,
input:focus,
select:focus {{

    border-color:
        #7b2cff !important;

    box-shadow:
        0 0 0 3px
        rgba(123,44,255,0.20) !important;
}}


/* ============================================================
   PLACEHOLDER
   ============================================================ */

textarea::placeholder,
input::placeholder {{

    color:
        #000000 !important;

    opacity:
        0.75 !important;
}}


/* ============================================================
   LABEL
   ============================================================ */

label,
label *,
label span {{

    color:
        #000000 !important;

    font-weight:
        700 !important;
}}


/* ============================================================
   ASK BUTTON
   ============================================================ */

#ask-button {{

    width:
        100% !important;

    min-height:
        52px !important;

    margin-top:
        12px !important;

    background:
        linear-gradient(
            135deg,
            #7b2cff,
            #ff2d95
        ) !important;

    color:
        #000000 !important;

    border:
        3px solid #000000 !important;

    border-radius:
        13px !important;

    font-size:
        17px !important;

    font-weight:
        900 !important;
}}


#ask-button *,
#ask-button span {{

    color:
        #000000 !important;
}}


/* ============================================================
   ANSWER
   ============================================================ */

#answer-box {{

    background:
        #ffffff !important;

    color:
        #000000 !important;

    border:
        2px solid #000000 !important;

    border-radius:
        16px !important;

    padding:
        25px !important;

    min-height:
        200px !important;

    box-shadow:
        0 8px 25px
        rgba(0,0,0,0.40) !important;
}}


#answer-box,
#answer-box *,
#answer-box p,
#answer-box li,
#answer-box span,
#answer-box strong {{

    color:
        #000000 !important;
}}


/* Keep answer headings colorful but text remains dark */
#answer-box h2,
#answer-box h3 {{

    color:
        #000000 !important;

    font-weight:
        900 !important;
}}


#answer-box li {{

    line-height:
        1.7 !important;
}}


/* ============================================================
   PAGE VIEWER
   ============================================================ */

#page-viewer {{

    background:
        #ffffff !important;

    color:
        #000000 !important;

    border:
        2px solid #000000 !important;

    border-radius:
        16px !important;

    padding:
        25px !important;

    min-height:
        350px !important;
}}


#page-viewer,
#page-viewer *,
#page-viewer p,
#page-viewer span,
#page-viewer strong,
#page-viewer h2,
#page-viewer h3 {{

    color:
        #000000 !important;
}}


/* ============================================================
   PAGE NAVIGATION BUTTONS
   ============================================================ */

.nav-button {{

    background:
        linear-gradient(
            135deg,
            #00d4ff,
            #00a8cc
        ) !important;

    color:
        #000000 !important;

    border:
        2px solid #000000 !important;

    border-radius:
        10px !important;

    min-height:
        45px !important;

    font-weight:
        900 !important;
}}


.nav-button *,
.nav-button span {{

    color:
        #000000 !important;
}}


.nav-button:hover {{

    background:
        linear-gradient(
            135deg,
            #7df3ff,
            #00d4ff
        ) !important;

    border-color:
        #000000 !important;

    color:
        #000000 !important;
}}


/* ============================================================
   PAGE NUMBER
   ============================================================ */

#page-number {{

    background:
        #ffffff !important;

    color:
        #000000 !important;

    border:
        2px solid #000000 !important;

    text-align:
        center !important;

    border-radius:
        10px !important;
}}


/* ============================================================
   FOOTER
   ============================================================ */

.footer-text,
.footer-text *,
.footer-text p,
.footer-text span {{

    text-align:
        center !important;

    color:
        #000000 !important;

    padding:
        10px !important;
}}


/* ============================================================
   FORCE ALL LETTERS BLACK
   This is the final override.
   ============================================================ */

html,
body,
.gradio-container,
.gradio-container *,
#main-container *,
.section-card *,
.markdown *,
button,
button *,
input,
input *,
textarea,
textarea *,
select,
select *,
label,
label *,
span,
p,
li,
strong,
em,
a,
h1,
h2,
h3,
h4,
h5,
h6 {{

    color:
        #000000 !important;
}}


/* ============================================================
   MOBILE
   ============================================================ */

@media (max-width: 768px) {{

    #main-container {{

        margin:
            8px !important;

        padding:
            12px !important;

        border-radius:
            15px !important;
    }}

    #college-header h1 {{

        font-size:
            25px !important;
    }}

    .question-button {{

        min-height:
            58px !important;
    }}

}}

"""


# ============================================================
# 29. GRADIO APPLICATION
# ============================================================

with gr.Blocks(
    title="MITT Thandavapura Q&A",
    css=CUSTOM_CSS,
    theme=gr.themes.Base(
        primary_hue="blue",
        secondary_hue="purple",
        neutral_hue="slate"
    )
) as demo:

    # ========================================================
    # MAIN
    # ========================================================

    with gr.Column(
        elem_id="main-container"
    ):

        # ====================================================
        # HEADER
        # ====================================================

        with gr.Column(
            elem_id="college-header"
        ):

            gr.Markdown(
                "# 🎓 Maharaja Institute of Technology Thandavapura"
            )

            gr.Markdown(
                "### MITT College Website Q&A Assistant"
            )

            gr.Markdown(
                "Explore college information, courses, "
                "admissions, placements, facilities and more."
            )

        # ====================================================
        # COLLEGE DETAILS
        # ====================================================

        with gr.Column(
            elem_classes="section-card details-card"
        ):

            gr.Markdown(
                "## 🏫 College Details"
            )

            gr.Markdown(
                """
**Maharaja Institute of Technology Thandavapura**

**Location:** Thandavapura, Mysuru District, Karnataka

**Official Website:** https://mitt.edu.in/

**Institution Type:** Engineering & Management Institute

**Website:** MITT Thandavapura
"""
            )

        # ====================================================
        # WEBSITE LOADING
        # ====================================================

        with gr.Column(
            elem_classes="section-card"
        ):

            gr.Markdown(
                "## 🌐 Official College Website"
            )

            gr.Markdown(
                "https://mitt.edu.in/",
                elem_classes="website-box"
            )

            load_button = gr.Button(
                "🚀 LOAD MITT WEBSITE",
                variant="primary",
                elem_id="load-button",
                size="lg"
            )

            status_box = gr.Markdown(
                """
## ⚪ Website Status

Website has not been loaded yet.

Click **LOAD MITT WEBSITE** to crawl the official MITT website.
""",
                elem_id="status-box"
            )

        # ====================================================
        # PAGE-BY-PAGE WEBSITE VIEW
        # ====================================================

        with gr.Column(
            elem_classes="section-card"
        ):

            gr.Markdown(
                "## 📚 Website Pages"
            )

            gr.Markdown(
                "After loading the website, browse the collected pages one by one."
            )

            with gr.Row():

                previous_button = gr.Button(
                    "⬅ Previous",
                    elem_classes="nav-button"
                )

                page_number = gr.Number(
                    value=1,
                    precision=0,
                    label="Page Number",
                    elem_id="page-number"
                )

                next_button = gr.Button(
                    "Next ➡",
                    elem_classes="nav-button"
                )

            page_viewer = gr.Markdown(
                """
## 📄 Page Viewer

Load the MITT website to view individual pages.
""",
                elem_id="page-viewer"
            )

            show_page_button = gr.Button(
                "📖 Show Page",
                elem_classes="nav-button"
            )

        # ====================================================
        # QUESTIONS
        # ====================================================

        with gr.Column(
            elem_classes="section-card"
        ):

            gr.Markdown(
                "## 💡 Suggested Questions"
            )

            gr.Markdown(
                "Select a question to automatically place it into the question box."
            )

            question_buttons = []

            for i in range(
                0,
                len(QUESTIONS),
                3
            ):

                with gr.Row():

                    for j in range(3):

                        index = i + j

                        if index < len(QUESTIONS):

                            button = gr.Button(
                                QUESTIONS[index],
                                elem_classes="question-button"
                            )

                            question_buttons.append(
                                button
                            )

        # ====================================================
        # QUESTION
        # ====================================================

        with gr.Column(
            elem_classes="section-card"
        ):

            gr.Markdown(
                "## ❓ Ask MITT"
            )

            question_box = gr.Textbox(

                label="Your Question",

                placeholder=(
                    "Select a suggested question or type your own question..."
                ),

                lines=3
            )

            ask_button = gr.Button(
                "🤖 GET ANSWER",
                variant="primary",
                elem_id="ask-button",
                size="lg"
            )

        # ====================================================
        # ANSWER
        # ====================================================

        with gr.Column(
            elem_classes="section-card"
        ):

            gr.Markdown(
                "## 🤖 Answer"
            )

            answer_box = gr.Markdown(
                """
## 👋 Welcome

Load the MITT website and select a question to get started.
""",
                elem_id="answer-box"
            )

        # ====================================================
        # FOOTER
        # ====================================================

        gr.Markdown(
            """
<div class="footer-text">

🎓 MITT Thandavapura Website Q&A Assistant

Information is collected from the official MITT website.

</div>
""",
            elem_classes="footer-text"
        )

    # ========================================================
    # LOAD WEBSITE
    # ========================================================

    load_button.click(
        fn=load_college,
        inputs=[],
        outputs=status_box
    )

    # ========================================================
    # PAGE VIEW
    # ========================================================

    show_page_button.click(
        fn=show_page,
        inputs=page_number,
        outputs=page_viewer
    )

    # ========================================================
    # NEXT PAGE
    # ========================================================

    next_button.click(
        fn=next_page,
        inputs=page_number,
        outputs=page_number
    ).then(
        fn=show_page,
        inputs=page_number,
        outputs=page_viewer
    )

    # ========================================================
    # PREVIOUS PAGE
    # ========================================================

    previous_button.click(
        fn=previous_page,
        inputs=page_number,
        outputs=page_number
    ).then(
        fn=show_page,
        inputs=page_number,
        outputs=page_viewer
    )

    # ========================================================
    # QUESTION BUTTONS
    # ========================================================

    for button in question_buttons:

        button.click(
            fn=select_question,
            inputs=button,
            outputs=question_box
        )

    # ========================================================
    # ASK QUESTION
    # ========================================================

    ask_button.click(
        fn=ask_question,
        inputs=question_box,
        outputs=answer_box
    )


# ============================================================
# 30. LAUNCH
# ============================================================

print()
print("=" * 70)
print("STARTING MITT THANDAVAPURA Q&A ASSISTANT")
print("=" * 70)

demo.launch(
    share=True,
    debug=True
)


/tmp/ipykernel_3319/1018773376.py:2419: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(



STARTING MITT THANDAVAPURA Q&A ASSISTANT
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://03da2aee633ddf7045.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
